### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="sdss_17",
    dataset_year="2022",
    domain_str="physics & astronomy",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/fedesoriano/stellar-classification-dataset-sdss17",
    download_description="""
We download the data from Kaggle.

mkdir -p local-data-warehouse/sdss_17/ && cd local-data-warehouse/sdss_17 && kaggle datasets download fedesoriano/stellar-classification-dataset-sdss17 && cd ../../ && unzip local-data-warehouse/sdss_17/stellar-classification-dataset-sdss17.zip -d local-data-warehouse/sdss_17 && rm local-data-warehouse/sdss_17/stellar-classification-dataset-sdss17.zip
""",
    # References
    academic_reference_bibtex=r"""@article{accetta2022seventeenth,
  title={The seventeenth data release of the Sloan Digital Sky Surveys: Complete release of MaNGA, MaStar, and APOGEE-2 data},
  author={Accetta, Katherine and Aerts, Conny and Aguirre, Victor Silva and Ahumada, Romina and Ajgaonkar, Nikhil and Ak, N Filiz and Alam, Shadab and Prieto, Carlos Allende and Almeida, Andres and Anders, Friedrich and others},
  journal={The Astrophysical Journal Supplement Series},
  volume={259},
  number={2},
  pages={35},
  year={2022},
  publisher={IOP Publishing}
}
""",
    academic_reference_bibtex_key="accetta2022seventeenth",
    license="Public Domain", # On Kaggle it says Data files © Original Authors which is under public domain.
    data_tags=["IID"],
    curation_comments="""
- We renamed the target feature.
- We dropped duplicates based on "obj_ID" to avoid target leakage from subgroups. 
- We dropped several (ID-like) meta-features that seem to be not part of the predictive task.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="ObjectType",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="ObjectType",
)

## Preprocessing

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dataset_mold.path}/star_classification.csv")

target_feature = "ObjectType"
df = df.rename(columns={"class": target_feature})

df = df.drop_duplicates(subset=["obj_ID"])

# Note: the following is from a very naive domain knowledge perspective
# and should be checked with domain experts. But otherwise, the predictive task
# might leak. So we are better safe than sorry.
df = df.drop(
    columns=[
        "obj_ID",  # should not be predictive, also has duplicates?
        "spec_obj_ID",  # ID
        "run_ID",  # should not be predictive but might indicate confounding noise
        "rerun_ID",  # constant
        "field_ID",  # might indicate clusters/subgroups of data?
        "MJD",  # date of observation, should not be predictive?
    ]
)

# Data is ordered, thus dist shift for original order. Shuffling the data removes this.
df = df.sample(frac=1, random_state=42).reset_index(drop=True)


cat_features = [
    "ObjectType",
    "plate",
    "fiber_ID",
]
df[cat_features] = df[cat_features].astype("category")

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()